# Check and create the clean DocVQA dataset

This supplementary notebook reproduces the project data-cleaning step. It runs on CPU without the source ZIP or ML libraries. It does not train a model or change checkpoints or evaluation results.

Input: the original `data` folder on Drive (200 training, 43 validation, and 57 evaluation questions). Remove training questions associated with documents in validation or evaluation; preserve those two splits. Expected output: 167 training questions/24 documents, 43 validation questions/12 documents, and 57 evaluation questions/12 documents.

If `data_clean` exists, only compare the data and check images; stop if they differ. Otherwise, copy the `data` folder and replace train.json with the filtered records. Unreferenced images may be retained, as in the previous cleaning run. Do not fabricate outputs from previous runs.


**Translation note:** This notebook is an English translation of the supplied notebook. Saved output messages were translated for readability; numerical results were preserved. Translation does not represent a new training or evaluation run.


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import json
from pathlib import Path

PROJECT = Path('/content/drive/MyDrive/Document_VQA_Project')
SOURCE = PROJECT / 'data'
DESTINATION = PROJECT / 'data_clean'
SPLITS = ('train', 'validation', 'evaluation')
EXPECTED_BEFORE = {'train': 200, 'validation': 43, 'evaluation': 57}
EXPECTED_AFTER = {'train': 167, 'validation': 43, 'evaluation': 57}
EXPECTED_DOCUMENTS = {'train': 24, 'validation': 12, 'evaluation': 12}

def read_rows(path):
    with path.open(encoding='utf-8') as handle:
        rows = json.load(handle)
    assert isinstance(rows, list), f'{path}: JSON must be a list'
    return rows

def document_ids(rows):
    return {str(row['document_id']) for row in rows}

def check_rows(rows, root, name):
    seen = set()
    for index, row in enumerate(rows):
        assert isinstance(row, dict), f'{name}[{index}]: sample is not a dict'
        for field in ('document_id', 'question_id', 'image_path', 'question', 'answers'):
            assert field in row, f'{name}[{index}]: missing {field}'
        assert row['document_id'] is not None and str(row['document_id']).strip()
        assert row['question_id'] is not None and str(row['question_id']).strip()
        qid = str(row['question_id'])
        assert qid not in seen, f'{name}: duplicate question_id {qid}'
        seen.add(qid)
        assert isinstance(row['question'], str) and row['question'].strip()
        assert isinstance(row['answers'], list) and row['answers']
        assert all(isinstance(answer, str) and answer.strip() for answer in row['answers'])
        assert isinstance(row['image_path'], str) and row['image_path'].strip()
        relative = Path(row['image_path'])
        assert not relative.is_absolute() and '..' not in relative.parts
        assert (root / relative).is_file(), f'{name}: missing image {relative}'

original = {}
for split in SPLITS:
    path = SOURCE / f'{split}.json'
    assert path.is_file(), f'Missing original data: {path}'
    original[split] = read_rows(path)
    assert len(original[split]) == EXPECTED_BEFORE[split], f'{split}: unexpected original sample count'
    check_rows(original[split], SOURCE, split)
    print(f'{split} original: {len(original[split])} questions, {len(document_ids(original[split]))} documents')


train original: 200 questions, 27 documents
validation original: 43 questions, 12 documents
evaluation original: 57 questions, 12 documents


In [3]:
validation_ids = document_ids(original['validation'])
evaluation_ids = document_ids(original['evaluation'])
assert not (validation_ids & evaluation_ids), 'Validation and evaluation share documents; check the splits'

excluded_ids = document_ids(original['train']) & (validation_ids | evaluation_ids)
assert excluded_ids == {'ffbg0227', 'ffhx0227', 'ffjw0228'}, f'Overlapping documents differ from the recorded run: {excluded_ids}'

clean = {
    'train': [row for row in original['train'] if str(row['document_id']) not in excluded_ids],
    'validation': original['validation'],
    'evaluation': original['evaluation'],
}
removed = len(original['train']) - len(clean['train'])
assert removed == 33, f'Expected to remove 33 questions, actually removed {removed}'
for split in SPLITS:
    assert len(clean[split]) == EXPECTED_AFTER[split]
    assert len(document_ids(clean[split])) == EXPECTED_DOCUMENTS[split]
for left, right in [('train', 'validation'), ('train', 'evaluation'), ('validation', 'evaluation')]:
    assert not (document_ids(clean[left]) & document_ids(clean[right]))

print('Documents removed from training:', sorted(excluded_ids))
print('Questions removed:', removed)
for split in SPLITS:
    print(f'{split}: {len(clean[split])} questions, {len(document_ids(clean[split]))} documents')
print('✅ The three splits have no overlapping document IDs')


Documents removed from training: ['ffbg0227', 'ffhx0227', 'ffjw0228']
Questions removed: 33
train: 167 questions, 24 documents
validation: 43 questions, 12 documents
evaluation: 57 questions, 12 documents
✅ The three splits have no overlapping document IDs


In [4]:
import shutil
import tempfile

receipt_path = SOURCE / 'receipts/evaluation.json'
assert receipt_path.is_file(), f'Missing receipt dataset: {receipt_path}'
receipt_rows = read_rows(receipt_path)
assert len(receipt_rows) == 40
assert len(document_ids(receipt_rows)) == 20
check_rows(receipt_rows, SOURCE / 'receipts', 'receipts')

if DESTINATION.exists():
    assert DESTINATION.is_dir(), f'{DESTINATION} is not a directory'
    for split in SPLITS:
        existing = read_rows(DESTINATION / f'{split}.json')
        assert existing == clean[split], f'{split}.json differs from the filtering result; refusing to overwrite'
        check_rows(existing, DESTINATION, split)
    assert read_rows(DESTINATION / 'receipts/evaluation.json') == receipt_rows, 'Existing receipt data differ from the original data'
    check_rows(receipt_rows, DESTINATION / 'receipts', 'receipts')
    print('✅ Existing data_clean matches the filtering result and contains all referenced images. No files were overwritten.')
else:
    # Create in a temporary directory and move to the destination only after validation.
    staging_parent = Path(tempfile.mkdtemp(prefix='vqa_clean_staging_', dir=str(PROJECT)))
    staging = staging_parent / 'data_clean'
    try:
        shutil.copytree(SOURCE, staging)
        with (staging / 'train.json').open('w', encoding='utf-8') as handle:
            json.dump(clean['train'], handle, ensure_ascii=False, indent=2)
        for split in SPLITS:
            assert read_rows(staging / f'{split}.json') == clean[split]
            check_rows(clean[split], staging, split)
        check_rows(receipt_rows, staging / 'receipts', 'receipts')
        assert not DESTINATION.exists(), 'The destination was created by another process; stopping to avoid overwriting'
        staging.rename(DESTINATION)
        print('✅ Created:', DESTINATION)
    finally:
        if staging_parent.exists():
            shutil.rmtree(staging_parent)

print('✅ Original data were preserved')
print('Receipts: 40 questions, 20 receipts')
print('Retraining is not required when using the trained clean checkpoint.')


✅ Existing data_clean matches the filtering result and contains all referenced images. No files were overwritten.
✅ Original data were preserved
Receipts: 40 questions, 20 receipts
Retraining is not required when using the trained clean checkpoint.


## Save in the project package

After a successful run, press Ctrl + S in Colab, download the `.ipynb`, and save it as `notebooks/Check_And_Clean_Dataset.ipynb`. Retain the actual verification output. This is a supplementary reproduction/checking notebook, not fabricated evidence of the initial cleaning run.

If a check reports different data or missing images, preserve the data and read the error before continuing. This notebook compares JSON contents and checks that images exist; it does not compare image hashes or decode every image. Do not run the training notebook for this check.
